## Model Approval

### Purpose

This notebook determines whether a registered model version should be promoted to production.

It evaluates logged metrics against predefined thresholds and conditionally assigns the `Champion` alias.

This is the second step in Stage 3 of the BYOM workflow (Evaluate → Approve → Deploy).

### What This Notebook Does

- Retrieves evaluation metrics from MLflow
- Compares metrics against approval thresholds
- Approves or rejects the model version
- Assigns or withholds the `Champion` alias
- Records approval outcome for auditability

This notebook enforces governance.  
It does **not** retrain or reevaluate the model.

In [0]:
%pip install --upgrade 'mlflow>=3.0.0'
%restart_python

In [0]:
# Widget parameters for job orchestration
dbutils.widgets.text("catalog_name", "pcl", "Catalog Name")
dbutils.widgets.text("schema_name", "byo_model", "Schema Name")
dbutils.widgets.text("model_name", "pyfunc_xgb", "Model Name")  # same as model_type when logging, or dl model name from artifacts.json
dbutils.widgets.text("model_version", "", "Model Version")
dbutils.widgets.text("accuracy_threshold", "0.25", "Accuracy Threshold") #change based on metrics
dbutils.widgets.text("f1_threshold", "0.10", "F1 Threshold") #change based on metrics

catalog = dbutils.widgets.get("catalog_name")
schema = dbutils.widgets.get("schema_name")
model_name = dbutils.widgets.get("model_name")
model_version = dbutils.widgets.get("model_version")
accuracy_threshold = float(dbutils.widgets.get("accuracy_threshold"))
f1_threshold = float(dbutils.widgets.get("f1_threshold"))

# Construct registered model name
registered_model_name = f"{catalog}.{schema}.{model_name}"

### Run approval

1. Specify required performance criteria (for example):
    - Minimum accuracy
    - Minimum F1 score
    - Maximum error rate

    Thresholds should reflect:
    - Business SLAs
    - Risk tolerance
    - Regulatory constraints (if applicable)

    Avoid hardcoding unrealistic values in customer engagements.

2. Pull evaluation metrics logged in the previous stage. Ensure:
    - Evaluation notebook completed successfully
    - Metrics are present for the target model version

3. Compare model metrics to defined thresholds.

    If criteria are met:
    - Mark model as approved

    If criteria are not met:
    - Reject promotion
    - Leave existing Champion unchanged

    Fail fast and clearly when thresholds are not satisfied.

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient
from datetime import datetime

mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()

# Resolve model version (from widget, upstream task, or latest)
if not model_version:
    try:
        model_version = dbutils.jobs.taskValues.get(taskKey="model_evaluation", key="model_version")
    except Exception:
        versions = client.search_model_versions(f"name='{registered_model_name}'")
        if not versions:
            raise ValueError(f"No versions found for model: {registered_model_name}")
        model_version = max(v.version for v in versions)
model_version = str(model_version)

# Get model version and evaluation metrics from tags
mv = client.get_model_version(registered_model_name, model_version)
tags = mv.tags
if tags.get("evaluation_status") != "completed":
    raise ValueError(f"Model version {model_version} has not been evaluated. Run 2_model_evaluation first.")

accuracy = float(tags.get("accuracy", "0"))
f1_weighted = float(tags.get("f1_weighted", "0"))

# Approval decision
approval_checks = {
    "accuracy": accuracy >= accuracy_threshold,
    "f1_weighted": f1_weighted >= f1_threshold,
}
approval_status = "approved" if all(approval_checks.values()) else "rejected"

# Set approval tags
client.set_model_version_tag(registered_model_name, model_version, "approval", approval_status)
client.set_model_version_tag(registered_model_name, model_version, "approval_timestamp", datetime.utcnow().isoformat())
client.set_model_version_tag(registered_model_name, model_version, "approval_thresholds", f"accuracy>={accuracy_threshold},f1>={f1_threshold}")

### Promote to Champion Alias (If Approved)

If approved:
- Assign the `Champion` alias to the evaluated version
- Ensure previous Champion is superseded

Downstream deployment notebooks reference the `Champion` alias, not version numbers, to maintain a controlled promotion path.

In [ ]:
# Set Champion / Challenger / Pending alias
if approval_status == "approved":
    try:
        current = client.get_model_version_by_alias(registered_model_name, "Champion")
        if current.version != model_version:
            client.set_registered_model_alias(registered_model_name, "Challenger", current.version)
    except Exception:
        pass
    client.set_registered_model_alias(registered_model_name, "Champion", model_version)
    print(f"Set {registered_model_name} version {model_version} as Champion")
else:
    client.set_registered_model_alias(registered_model_name, "Pending", model_version)
    print(f"Set {registered_model_name} version {model_version} as Pending (rejected)")

dbutils.jobs.taskValues.set(key="model_name", value=registered_model_name)
dbutils.jobs.taskValues.set(key="model_version", value=model_version)
dbutils.jobs.taskValues.set(key="approval_status", value=approval_status)

print(f"Approval completed: {registered_model_name} v{model_version} -> {approval_status}")
print(f"  Accuracy: {accuracy:.4f} (threshold {accuracy_threshold})  F1: {f1_weighted:.4f} (threshold {f1_threshold})")